# 📘 W2 Python Lab — 기술적 지표 (RSI · MA · 볼린저)

**n8n W2 강의의 Python 버전.** Code 노드의 JavaScript 로직을 Python으로 재구현합니다.

## 학습 목표
- `yfinance`로 60일 OHLC 대량 수집
- numpy·pandas로 RSI·MA·볼린저밴드 계산
- `matplotlib`로 차트 시각화
- 3개 지표 조합으로 매매 신호 생성

## 🛠 사전 준비

```bash
pip install yfinance pandas numpy matplotlib
```

> 💡 **n8n과 비교:** n8n Code 노드의 JavaScript 계산 로직 = Python에서는 pandas vector 연산으로 훨씬 간결하고 빠르게 구현.

## 1. 데이터 로딩 — yfinance

`yfinance`는 Yahoo Finance 비공식 Python 클라이언트. 60일 OHLC를 한 번의 호출로 가져올 수 있습니다.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def load_ohlc(ticker: str, period: str = '3mo') -> pd.DataFrame:
    """yfinance로 OHLC 데이터 로딩.
    
    period: '1mo', '3mo', '6mo', '1y', '5y', 'max'
    """
    df = yf.Ticker(ticker).history(period=period)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    df.index = df.index.tz_localize(None)  # 타임존 제거
    return df

# 테스트: 애플 3개월 OHLC
aapl = load_ohlc('AAPL', period='3mo')
print(f'Rows: {len(aapl)}')
aapl.tail()

## 2. RSI (Relative Strength Index) 계산

n8n Code 노드의 JS 로직과 1:1 대응되는 Python 구현.

**공식:**
- `RSI = 100 - (100 / (1 + RS))`
- `RS = 평균상승 / 평균하락`  (기본 14일)

In [ ]:
def calc_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    """RSI 계산 (Wilder's Smoothing)."""
    delta = close.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    
    # Wilder's smoothing (EMA와 유사)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

aapl['RSI14'] = calc_rsi(aapl['Close'], 14)

# RSI 해석
latest_rsi = aapl['RSI14'].iloc[-1]
if latest_rsi > 70:
    signal = '🔴 과매수 (overbought)'
elif latest_rsi < 30:
    signal = '🟢 과매도 (oversold)'
else:
    signal = '⚪ 중립'

print(f'현재 RSI(14): {latest_rsi:.2f} → {signal}')
aapl[['Close', 'RSI14']].tail()

## 3. 이동평균 (MA · EMA)

n8n JS에서는 for 루프로 계산했지만 pandas는 `.rolling()` 한 줄.

In [ ]:
# 단순 이동평균
aapl['MA5'] = aapl['Close'].rolling(window=5).mean()
aapl['MA20'] = aapl['Close'].rolling(window=20).mean()
aapl['MA60'] = aapl['Close'].rolling(window=60).mean()

# 지수 이동평균 (최근 데이터에 더 가중치)
aapl['EMA20'] = aapl['Close'].ewm(span=20, adjust=False).mean()

# 골든크로스/데드크로스 감지
def detect_cross(series_short: pd.Series, series_long: pd.Series) -> str:
    """어제와 오늘 비교해 크로스 감지."""
    if len(series_short) < 2 or series_short.iloc[-2:].isna().any():
        return 'INSUFFICIENT_DATA'
    
    yesterday_diff = series_short.iloc[-2] - series_long.iloc[-2]
    today_diff = series_short.iloc[-1] - series_long.iloc[-1]
    
    if yesterday_diff < 0 and today_diff > 0:
        return '🟢 GOLDEN_CROSS'
    elif yesterday_diff > 0 and today_diff < 0:
        return '🔴 DEAD_CROSS'
    return '⚪ NO_CROSS'

cross_signal = detect_cross(aapl['MA5'], aapl['MA20'])
print(f'MA5 vs MA20: {cross_signal}')
aapl[['Close', 'MA5', 'MA20', 'MA60', 'EMA20']].tail()

## 4. 볼린저 밴드

MA ± 2σ. 가격이 상단에 닿으면 과매수, 하단에 닿으면 과매도.  
%B 지표로 밴드 내 위치도 계산.

In [ ]:
def calc_bollinger(close: pd.Series, period: int = 20, std_mult: float = 2.0) -> pd.DataFrame:
    """볼린저 밴드 + %B + Bandwidth 계산."""
    ma = close.rolling(window=period).mean()
    std = close.rolling(window=period).std()
    
    upper = ma + std_mult * std
    lower = ma - std_mult * std
    
    # %B: 현재 가격이 밴드 내 어디인지 (0 = 하단, 1 = 상단)
    percent_b = (close - lower) / (upper - lower)
    
    # Bandwidth: 밴드 폭 (변동성 지표)
    bandwidth = (upper - lower) / ma
    
    return pd.DataFrame({
        'BB_Mid': ma,
        'BB_Upper': upper,
        'BB_Lower': lower,
        'BB_PercentB': percent_b,
        'BB_Bandwidth': bandwidth
    })

bb = calc_bollinger(aapl['Close'])
aapl = pd.concat([aapl, bb], axis=1)

# 최신 상태 해석
latest = aapl.iloc[-1]
pb = latest['BB_PercentB']

if pb > 1.0:
    bb_signal = '🔴 상단 밴드 상향 돌파 (강한 상승 모멘텀)'
elif pb > 0.8:
    bb_signal = '🟡 상단 접근 (과매수 경계)'
elif pb < 0:
    bb_signal = '🟢 하단 밴드 하향 이탈 (극단적 과매도)'
elif pb < 0.2:
    bb_signal = '🟢 하단 접근 (과매도 경계)'
else:
    bb_signal = '⚪ 중간 영역'

print(f'Close: ${latest["Close"]:.2f}')
print(f'BB: [{latest["BB_Lower"]:.2f}, {latest["BB_Upper"]:.2f}]')
print(f'%B: {pb:.3f} → {bb_signal}')

## 5. 종합 매매 신호 생성

n8n Code 노드의 Verdict 로직을 Python으로. 3개 지표를 조합해 BUY/SELL/HOLD 결정.

In [ ]:
def generate_signal(df: pd.DataFrame) -> dict:
    """RSI + MA + BB 종합으로 신호 생성."""
    latest = df.iloc[-1]
    
    score = 0
    reasons = []
    
    # RSI 점수 (-2 ~ +2)
    rsi = latest['RSI14']
    if rsi < 30:
        score += 2
        reasons.append(f'RSI 과매도({rsi:.1f})')
    elif rsi < 45:
        score += 1
        reasons.append(f'RSI 매수 영역({rsi:.1f})')
    elif rsi > 70:
        score -= 2
        reasons.append(f'RSI 과매수({rsi:.1f})')
    elif rsi > 55:
        score -= 1
        reasons.append(f'RSI 매도 영역({rsi:.1f})')
    
    # MA 점수 (-1 ~ +1)
    if latest['MA5'] > latest['MA20']:
        score += 1
        reasons.append('단기MA > 장기MA (상승 추세)')
    else:
        score -= 1
        reasons.append('단기MA < 장기MA (하락 추세)')
    
    # 볼린저 점수 (-2 ~ +2)
    pb = latest['BB_PercentB']
    if pb < 0.2:
        score += 2
        reasons.append(f'BB 하단 접근(%B={pb:.2f})')
    elif pb > 0.8:
        score -= 2
        reasons.append(f'BB 상단 접근(%B={pb:.2f})')
    
    # 최종 판단
    if score >= 3:
        verdict, conf = 'BUY', min(5, abs(score))
    elif score <= -3:
        verdict, conf = 'SELL', min(5, abs(score))
    elif score >= 1:
        verdict, conf = 'WATCH_BUY', 2
    elif score <= -1:
        verdict, conf = 'WATCH_SELL', 2
    else:
        verdict, conf = 'HOLD', 1
    
    return {
        'verdict': verdict,
        'confidence': conf,
        'score': score,
        'price': round(latest['Close'], 2),
        'reasons': reasons
    }

signal = generate_signal(aapl)
print(f"\n📊 AAPL 매매 신호")
print(f"  판단: {signal['verdict']} (신뢰도 {signal['confidence']}/5)")
print(f"  점수: {signal['score']}")
print(f"  근거:")
for r in signal['reasons']:
    print(f'    • {r}')

## 6. 시각화 — 차트로 확인

n8n에서는 Chart-IMG API로 외부 차트를 받아왔지만, Python은 matplotlib로 직접 그립니다.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

# 상단: 가격 + 이동평균 + 볼린저
ax1.plot(aapl.index, aapl['Close'], label='Close', color='black', linewidth=1.5)
ax1.plot(aapl.index, aapl['MA20'], label='MA20', color='blue', alpha=0.7)
ax1.plot(aapl.index, aapl['MA60'], label='MA60', color='red', alpha=0.5)
ax1.fill_between(aapl.index, aapl['BB_Lower'], aapl['BB_Upper'], 
                   alpha=0.15, color='gray', label='BB (±2σ)')
ax1.set_title(f'AAPL — Price + MA + Bollinger Bands', fontsize=13, fontweight='bold')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(alpha=0.3)

# 하단: RSI
ax2.plot(aapl.index, aapl['RSI14'], color='purple', linewidth=1)
ax2.axhline(70, color='red', linestyle='--', alpha=0.5, label='Overbought (70)')
ax2.axhline(30, color='green', linestyle='--', alpha=0.5, label='Oversold (30)')
ax2.axhline(50, color='gray', linestyle=':', alpha=0.3)
ax2.fill_between(aapl.index, 70, 100, alpha=0.1, color='red')
ax2.fill_between(aapl.index, 0, 30, alpha=0.1, color='green')
ax2.set_ylabel('RSI(14)', fontsize=10)
ax2.set_ylim(0, 100)
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('aapl_technical.png', dpi=100, bbox_inches='tight')
plt.show()
print('\n✓ 차트 저장: aapl_technical.png')

## 7. 여러 종목 배치 분석

n8n의 Split In Batches 노드 역할. 포트폴리오 전체를 한 번에 평가.

In [ ]:
def batch_analyze(tickers: list[str], period: str = '3mo') -> pd.DataFrame:
    """여러 종목을 한 번에 분석해 신호 DataFrame 반환."""
    results = []
    
    for ticker in tickers:
        try:
            df = load_ohlc(ticker, period)
            df['RSI14'] = calc_rsi(df['Close'])
            df['MA5'] = df['Close'].rolling(5).mean()
            df['MA20'] = df['Close'].rolling(20).mean()
            df = pd.concat([df, calc_bollinger(df['Close'])], axis=1)
            
            sig = generate_signal(df)
            sig['ticker'] = ticker
            results.append(sig)
        except Exception as e:
            print(f'Error {ticker}: {e}')
    
    df_results = pd.DataFrame(results)
    if not df_results.empty:
        df_results = df_results[['ticker', 'verdict', 'confidence', 'score', 'price']]
        df_results = df_results.sort_values('score', ascending=False)
    return df_results

# 테스트: 관심 포트폴리오
portfolio = ['AAPL', 'MSFT', 'GOOGL', 'TSLA', 'NVDA', 'META', 'AMZN']
results = batch_analyze(portfolio)
print('\n📊 포트폴리오 신호 요약 (점수 내림차순):')
print(results.to_string(index=False))

## 🎯 과제

1. **MACD 추가** — `EMA12 - EMA26`의 Signal 라인 교차로 매매 신호
2. **한국 종목 테스트** — `005930.KS` (삼성전자), `035420.KS` (네이버)로 동일 분석
3. **backtest 간이 구현** — 과거 신호대로 매매했으면 수익률이 어땠을지 계산
   ```python
   # 힌트: RSI < 30이면 매수, > 70이면 매도한 가상 포지션
   df['position'] = np.where(df['RSI14'] < 30, 1, 
                             np.where(df['RSI14'] > 70, -1, 0))
   df['returns'] = df['Close'].pct_change() * df['position'].shift(1)
   print(f'누적 수익률: {(1 + df["returns"]).prod() - 1:.2%}')
   ```
4. **CSV 내보내기** — `batch_analyze` 결과를 W1 CSV와 합쳐 통합 일지 구축

## 🔄 n8n vs Python 비교

| 작업 | n8n Code 노드 | Python pandas |
|---|---|---|
| RSI 계산 | 20줄 JS 루프 | 4줄 벡터 연산 |
| MA 계산 | 10줄 JS 루프 | `.rolling().mean()` |
| 배치 분석 | Split In Batches + 수작업 병합 | for + DataFrame |
| 차트 생성 | 외부 API | matplotlib 자체 |

**언제 어느 것을 쓰는가:**
- **n8n**: 스케줄링·알림·외부 API 연동이 주인 경우 (자동화 워크플로)
- **Python**: 복잡한 계산·백테스트·대량 분석이 주인 경우 (데이터 작업)
- **두 가지를 결합**: Python으로 분석 → 결과 CSV를 n8n이 읽어 알림 발송